An example of schema evolution error that was correctly handled by the AutoLoader (Encountered unknown fields during parsing: {"bidding_zone":"ES"}, **which can be fixed by an automatic retry: true**)
![image_1784141485879.png](./image_1784141485879.png "image_1784141485879.png")

After re-run the files from ES bidding zone are correctly ingested to bronze layer:

In [0]:
%sql
SELECT *
FROM dbr_dev.gabrielajaniszews786_bronze.entsoe_prices
WHERE bidding_zone = 'ES'
LIMIT 10

### Experimenting with different trigger types

**Streaming stats — Auto Loader (processingTime trigger)**
The stream runs as micro-batches on a fixed trigger interval, so both charts are spiky by design rather than flat. Processing rate sits consistently above input rate, which means Spark is draining data faster than it arrives — the consumer is keeping up and catching up on backlog, so no lag accumulates.
The processing-rate spikes line up with bursts of new files from the producer: the fetch job writes files in batches, Auto Loader discovers a chunk at once, and each trigger crunches the accumulated files in a single burst before going quiet until the next batch. The large initial spike is the startup backlog being cleared in the first batch.
Batch duration follows the same pattern, rising on heavier batches and settling on lighter ones. As long as batch duration stays under the trigger interval, batches don't queue up and the stream stays stable. The input-below-processing relationship is the healthy signal here; the inverse would indicate backpressure and a growing backlog.

![image_1784143512523.png](./image_1784143512523.png "image_1784143512523.png")



**Note — trigger(once=True) reporting no new data**
Ran a one-time load with trigger(once=True) and it kept starting and stopping instantly: status = Stopped, isDataAvailable = False, lastProgress = None, no exception. Files were clearly present in landing and one zone (LT) was missing from the target table, so the "nothing to do" result was misleading.

Root cause was a leftover continuous stream (processingTime) still active on the same checkpointLocation. Two queries sharing one checkpoint conflict: the background stream owned/advanced the checkpoint state, so the once query saw everything as already processed and exited without a batch. Once I stopped the active queries (spark.streams.active → stop()), the next run picked up only the outstanding LT files and left the existing rows untouched — clean incremental load, no full reload needed.

**Takeaway**: always stop running queries before launching another against the same checkpoint, and check spark.streams.active when a trigger reports no data even though new files exist. isDataAvailable: False means "checkpoint thinks it's caught up," not "no files on disk."